# Compare GPDC regression backends

This small experiment asks which settings are needed to reproduce the current
fixed-kernel GPDC with GPyTorch, and whether CUDA helps at the existing record
length. It reuses the three original uniform records from experiment 02.

There are only six fixed conditional tests: two directions for each process.
This is not a PCMCI+ graph search, a chronology experiment, or a calibrated
false-positive study. Production estimators and saved results are not changed.

Select the Python interpreter in `.venv-gpdc`. Its pinned dependencies are in
`agent/requirements-gpdc.txt`; the production `.venv` is separate.

References: [native GPDC](https://jakobrunge.github.io/tigramite/_modules/tigramite/independence_tests/gpdc.html),
[native GPDCtorch](https://jakobrunge.github.io/tigramite/_modules/tigramite/independence_tests/gpdc_torch.html),
and [GPyTorch exact GP regression](https://docs.gpytorch.ai/en/v1.13/examples/01_Exact_GPs/Simple_GP_Regression.html).


## 1. Settings and numerical environment
Keep all records at their original 20-year resolution. One numerical CPU thread
is used; CUDA timings include transfers and synchronization. Times are single
passes after a small warm-up, not production-throughput benchmarks.


In [ ]:
import os
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS", "BLIS_NUM_THREADS"]:
    os.environ[name] = "1"

from pathlib import Path
from functools import partial
from importlib.metadata import version
import json
import platform
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import gpytorch
from IPython.display import display
from threadpoolctl import threadpool_limits
from tigramite.independence_tests.gpdc import GPDC
from tigramite.independence_tests.gpdc_torch import GPDCtorch

ROOT = Path.cwd().resolve()
if not (ROOT / "toolbox").is_dir():
    raise RuntimeError("Run this notebook with the project root as working directory.")
STEM = "02_5_compare_gpdc_backends"
DATA_DIR = ROOT / "data/processed/02_check_causal_pipeline"
OUT = ROOT / "data/processed" / STEM
FIG = ROOT / "figures" / STEM
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

PROCESSES = ["linear", "nonlinear", "independent"]
DIRECTIONS = [("x", "y"), ("y", "x")]
DT = 0.020             # Uniform step, kyr: 20 years.
START = 10             # Discard ten initial rows, matching the main effective T=1391.
NULL_SEED = 20261309   # Existing native GPDC reference-null seed.
NULL_SAMPLES = 10000   # Reuse the completed native reference distribution.
RUN_STOCK_TORCH = True # Native float32, trained GPDCtorch; six tests, not a parameter sweep.
ALIGN_ATOL = 1e-8      # Diagnostic absolute tolerance for standardized regression outputs.
ALIGN_RTOL = 1e-8      # Diagnostic relative tolerance; report exact equality separately.

torch.set_num_threads(1)
torch.manual_seed(NULL_SEED)
thread_limits = threadpool_limits(limits=1)
HAS_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if HAS_CUDA else "cpu"
packages = {name: version(name) for name in
            ["numpy", "scipy", "pandas", "scikit-learn", "tigramite", "dcor",
             "torch", "gpytorch", "linear-operator"]}
environment = {"packages": packages, "platform": platform.platform(),
               "cuda_available": HAS_CUDA, "torch_cuda": torch.version.cuda,
               "device": torch.cuda.get_device_name(0) if HAS_CUDA else "CPU",
               "cpu_threads": torch.get_num_threads()}
display(pd.Series(environment, name="Environment"))
if not HAS_CUDA:
    print("CUDA is unavailable: the aligned CPU comparison still runs; no GPU timing is claimed.")

from toolbox.plotting import add_panel_labels


## 2. Reuse the original uniform inputs
No new processes, sampling masks or chronologies are generated. Rows advance in
physical time while age decreases from 29 to 1 kyr BP.


In [ ]:
series = {}
for process in PROCESSES:
    path = DATA_DIR / f"r0_{process}_original_uniform_series.csv"
    frame = pd.read_csv(path)
    assert len(frame) == 1401 and np.isfinite(frame[["time", "age", "x", "y"]]).all().all()
    np.testing.assert_allclose(np.diff(frame["time"]), DT, rtol=0, atol=1e-12)
    np.testing.assert_allclose(np.diff(frame["age"]), -DT, rtol=0, atol=1e-12)
    series[process] = frame

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True, constrained_layout=True)
for ax, (process, frame) in zip(axes, series.items()):
    ax.plot(frame["age"], frame["x"], label="X", linewidth=0.8)
    ax.plot(frame["age"], frame["y"], label="Y", linewidth=0.8, alpha=0.8)
    ax.set_ylabel(process.capitalize())
axes[0].legend(ncol=2)
axes[-1].set_xlabel("Age (kyr BP)")
axes[-1].invert_xaxis()
add_panel_labels(axes)
fig.savefig(FIG / "01_regular_inputs.png", dpi=180)
fig.savefig(FIG / "01_regular_inputs.pdf")
plt.show()


## 3. Specify six conditional tests
For X to Y we test $X_{t-1}$ against $Y_t$, conditional on
$[X_{t-2},Y_{t-1}]$; the other direction swaps X and Y. The condition set is
fixed to isolate backend differences. A nonempty condition set is essential:
without Z, GPDC skips GP regression and would not test the GPU component.


In [ ]:
cases = {}
for process, frame in series.items():
    t = np.arange(START, len(frame))
    for source, target in DIRECTIONS:
        source_values = frame[source].to_numpy()
        target_values = frame[target].to_numpy()
        array = np.vstack([source_values[t - 1], target_values[t],
                           source_values[t - 2], target_values[t - 1]])
        key = f"{process}_{source}_to_{target}"
        cases[key] = {"array": array, "age": frame["age"].to_numpy()[t],
                      "process": process, "source": source, "target": target}
        assert array.shape == (4, 1391)
display(pd.DataFrame([{"case": key, "T": value["array"].shape[1],
                       "condition_count": 2} for key, value in cases.items()]))


## 4. Hold the native significance calculation fixed
All backends use the existing 10,000-draw native null distribution at T=1391.
Its metadata must match this Windows numerical environment. The native
analytic p value is the fraction of null statistics strictly above the score;
zero means no exceedance among these draws, not a probability known to be zero.


In [ ]:
null_path = ROOT / "data/cache" / f"gpdc_T1391_n{NULL_SAMPLES}_seed{NULL_SEED}.npz"
null_settings = json.loads(null_path.with_suffix(".json").read_text(encoding="utf-8"))
assert null_settings["sample_sizes"] == [1391]
assert null_settings["sig_samples"] == NULL_SAMPLES and null_settings["seed"] == NULL_SEED
assert null_settings["platform"] == platform.system()
assert null_settings["machine"] == platform.machine()
for name, expected in null_settings["packages"].items():
    assert version(name) == expected, f"Reference-null package mismatch: {name}"

native = GPDC(significance="analytic", sig_samples=NULL_SAMPLES, seed=NULL_SEED,
              gp_params={"optimizer": None}, null_dist_filename=str(null_path))
stock = GPDCtorch(significance="analytic", sig_samples=NULL_SAMPLES, seed=NULL_SEED,
                  null_dist_filename=str(null_path)) if RUN_STOCK_TORCH else None
assert len(native.gauss_pr.null_dists[1391]) == NULL_SAMPLES
assert np.isfinite(native.gauss_pr.null_dists[1391]).all()

# Warm the existing distance-correlation path and a tiny CUDA operation only.
native._get_dcorr(cases["linear_x_to_y"]["array"][:2].copy())
if HAS_CUDA:
    warm_matrix = torch.eye(16, dtype=torch.float64, device="cuda")
    torch.linalg.cholesky(warm_matrix)
    torch.cuda.synchronize()


## 5. Make the migration settings explicit
The stock implementation is a different fit: float32, learned constant mean,
kernel scale, length scale and likelihood noise, with up to 50 native training
iterations. Its fitted model is local to the native method and is not exposed.

The aligned experiment instead uses float64, zero mean, RBF scale=1 and
length scale=1, likelihood noise=1, and no training. Noise enters the training
covariance; prediction uses the latent posterior mean, matching sklearn's
RBF + WhiteKernel cross-covariance convention. Extra jitter and approximate
solves are disabled here. This local comparison is not a production adapter.


In [ ]:
settings_table = pd.DataFrame({
    "setting": ["dtype", "mean", "RBF length scale", "kernel scale", "noise variance",
                "hyperparameter training", "linear solve", "prediction", "GPDC statistic"],
    "current_GPDC": ["float64", "zero", 1, 1, 1, "disabled", "dense Cholesky",
                     "latent mean", "native empirical-CDF + AVL distance correlation"],
    "stock_GPDCtorch": ["float32", "learned constant", "learned", "learned", "learned",
                        "native default, up to 50 iterations", "native GPyTorch settings",
                        "latent mean", "same native statistic"],
    "aligned_GPyTorch": ["float64", "zero", 1, 1, 1, "disabled", "dense Cholesky; no added jitter",
                         "latent mean", "same native statistic"],
})
display(settings_table)

class FixedGP(gpytorch.models.ExactGP):
    """The existing fixed GP model expressed through the native GPyTorch API."""
    def __init__(self, x, y, likelihood):
        super().__init__(x, y, likelihood)
        self.mean_module = gpytorch.means.ZeroMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(self.mean_module(x), self.covar_module(x))


def aligned_residuals(array, target_var, *, device, return_means=True):
    """Preserve native in-place standardization, then evaluate the fixed GP."""
    # Native GPDC standardizes all rows again for the second residual call.
    array[:] = array - array.mean(axis=1).reshape(len(array), 1)
    std = array.std(axis=1)
    for i in range(len(array)):
        if std[i] != 0:
            array[i] = array[i] / std[i]
    x = torch.tensor(array[2:].T.copy(), dtype=torch.float64, device=device)
    y = torch.tensor(array[target_var].copy(), dtype=torch.float64, device=device)
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device=device, dtype=torch.float64)
    model = FixedGP(x, y, likelihood).to(device=device, dtype=torch.float64)
    # Initialize after conversion to double; default raw Torch parameters are not 1.
    model.covar_module.base_kernel.lengthscale = 1.0
    model.covar_module.outputscale = 1.0
    likelihood.noise = 1.0
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    model.eval()
    likelihood.eval()
    with torch.no_grad(), \
         gpytorch.settings.max_cholesky_size(10000), \
         gpytorch.settings.fast_computations(covar_root_decomposition=False, log_prob=False, solves=False), \
         gpytorch.settings.fast_pred_var(False), \
         gpytorch.settings.cholesky_jitter(float_value=0.0, double_value=0.0):
        mean = model(x).mean.detach().cpu().numpy()
    residual = array[target_var] - mean
    return (residual, mean) if return_means else residual


## 6. Run the same regression and statistic sequence
For each test, copy the input once, fit row 0 and then row 1 on the same working
array, and use Tigramite's unchanged distance-correlation and p-value methods.
This preserves the native sequence of standardization. Save full regression
vectors, not only whether p crosses a threshold. Failures remain explicit.


In [ ]:
def evaluate_case(case, residual_function, *, uses_cuda):
    work = case["array"].copy()
    means, residuals = [], []
    if uses_cuda:
        torch.cuda.synchronize()
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        for target_var in [0, 1]:
            residual, mean = residual_function(work, target_var, return_means=True)
            residuals.append(residual.copy())
            means.append(mean.copy())
        residuals = np.asarray(residuals)
        means = np.asarray(means)
        score = native._get_dcorr(residuals)
        p_value = native.get_analytic_significance(score, T=work.shape[1], dim=4,
                                                  xyz=np.array([0, 1, 2, 2]))
    if uses_cuda:
        torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    assert np.isfinite(means).all() and np.isfinite(residuals).all()
    assert np.isfinite(score) and np.isfinite(p_value) and 0 <= p_value <= 1
    warning_text = sorted(set(str(item.message) for item in caught))
    return {"means": means, "residuals": residuals, "score": float(score),
            "p_value": float(p_value), "seconds": seconds, "warnings": warning_text}


backends = {"GPDC_cpu": (native._get_single_residuals, False)}
if RUN_STOCK_TORCH:
    backends["GPDCtorch_stock"] = (stock._get_single_residuals, HAS_CUDA)
backends["aligned_cpu"] = (partial(aligned_residuals, device="cpu"), False)
if HAS_CUDA:
    backends["aligned_cuda"] = (partial(aligned_residuals, device="cuda"), True)

results, rows = {}, []
for case_name, case in cases.items():
    for backend, (residual_function, uses_cuda) in backends.items():
        torch.manual_seed(NULL_SEED)
        try:
            result = evaluate_case(case, residual_function, uses_cuda=uses_cuda)
            results[(case_name, backend)] = result
            row = {"case": case_name, "backend": backend, "status": "ok", "error": "",
                   **{key: result[key] for key in ["score", "p_value", "seconds"]},
                   "warnings": " | ".join(result["warnings"])}
        except Exception as error:
            row = {"case": case_name, "backend": backend, "status": "failed",
                   "error": f"{type(error).__name__}: {error}"}
        rows.append(row)
        print(case_name, backend, row["status"], f"{row.get('seconds', float('nan')):.3f} s", flush=True)

summary = pd.DataFrame(rows)
summary.to_csv(OUT / "backend_summary.csv", index=False)
assert summary.loc[summary.backend == "GPDC_cpu", "status"].eq("ok").all(), "Reference failures saved in backend_summary.csv."
# One direct native call checks that exposing means/residuals did not change the reference statistic.
direct_score = native.get_dependence_measure(cases["linear_x_to_y"]["array"].copy(), np.array([0, 1, 2, 2]))
assert direct_score == results[("linear_x_to_y", "GPDC_cpu")]["score"]
display(summary[["case", "backend", "status", "score", "p_value", "seconds"]].round(7))


## 7. Compare full regression outputs
The tolerances below are diagnostic targets for this small migration check,
not acceptance of a new production backend. Exact numerical equality and
tolerance agreement are reported separately. A matching score alone does not
prove that every fitted value is identical.


In [ ]:
comparisons = []
arrays = {}
for (case_name, backend), result in results.items():
    reference = results[(case_name, "GPDC_cpu")]
    mean_error = result["means"] - reference["means"]
    residual_error = result["residuals"] - reference["residuals"]
    comparisons.append({
        "case": case_name, "backend": backend,
        "mean_max_abs_error": float(np.max(np.abs(mean_error))),
        "residual_max_abs_error": float(np.max(np.abs(residual_error))),
        "residual_rmse": float(np.sqrt(np.mean(residual_error ** 2))),
        "regression_exact_equal": bool(np.array_equal(result["means"], reference["means"])
                                        and np.array_equal(result["residuals"], reference["residuals"])),
        "regression_within_tolerance": bool(np.allclose(result["means"], reference["means"],
                                                         rtol=ALIGN_RTOL, atol=ALIGN_ATOL)
                                            and np.allclose(result["residuals"], reference["residuals"],
                                                            rtol=ALIGN_RTOL, atol=ALIGN_ATOL)),
        "score_difference": result["score"] - reference["score"],
        "p_difference": result["p_value"] - reference["p_value"],
    })
    arrays[f"{case_name}__{backend}__means"] = result["means"]
    arrays[f"{case_name}__{backend}__residuals"] = result["residuals"]
for case_name, case in cases.items():
    arrays[f"{case_name}__input"] = case["array"]
    arrays[f"{case_name}__age"] = case["age"]
comparison = pd.DataFrame(comparisons)
display(comparison)

summary.to_csv(OUT / "backend_summary.csv", index=False)
comparison.to_csv(OUT / "regression_comparison.csv", index=False)
settings_table.to_csv(OUT / "parameter_mapping.csv", index=False)
np.savez_compressed(OUT / "regression_arrays.npz", **arrays)
run_info = {"environment": environment, "inputs": [str((DATA_DIR / f"r0_{p}_original_uniform_series.csv").relative_to(ROOT)) for p in PROCESSES],
            "test_definition": "source[t-1] independent of target[t] conditional on source[t-2], target[t-1]",
            "T": 1391, "case_count": len(cases), "backend_count": len(backends),
            "null_file": str(null_path.relative_to(ROOT)), "null_metadata": null_settings,
            "seed": NULL_SEED, "rtol": ALIGN_RTOL, "atol": ALIGN_ATOL,
            "timing": "One pass per case/backend, includes transfers and synchronization; not production throughput.",
            "failed_tests": summary.loc[summary.status != "ok", ["case", "backend", "error"]].to_dict("records")}
(OUT / "run_info.json").write_text(json.dumps(run_info, indent=2), encoding="utf-8")


## 8. Inspect the migration visually
The first plot compares the six statistics and regression differences. The
second shows target residuals in one nonlinear test; matching aligned curves
should overlap the reference, while the difference panel exposes small errors.


In [ ]:
labels = [key.replace("independent", "indep.").replace("_to_", "->").replace("_", " ") for key in cases]
positions = np.arange(len(cases))
colors = {"GPDC_cpu": "#333333", "GPDCtorch_stock": "#D95F02",
          "aligned_cpu": "#1B9E77", "aligned_cuda": "#7570B3"}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
for backend in backends:
    part = summary[summary.backend == backend].set_index("case").reindex(cases)
    comp = comparison[comparison.backend == backend].set_index("case").reindex(cases)
    axes[0].plot(positions, part["score"], "o-", label=backend, color=colors[backend], alpha=0.8)
    axes[1].plot(positions, part["p_value"], "o-", color=colors[backend], alpha=0.8)
    if backend != "GPDC_cpu":
        axes[2].semilogy(positions, np.maximum(comp["residual_max_abs_error"], 1e-17),
                         "o-", color=colors[backend], alpha=0.8)
for ax, title in zip(axes, ["GPDC statistic", "Native empirical p", "Maximum residual difference"]):
    ax.set_title(title)
    ax.set_xticks(positions, labels, rotation=55, ha="right")
    ax.grid(alpha=0.2)
axes[0].legend(fontsize=8)
axes[1].set_ylim(-0.02, 1.02)
axes[2].axhline(ALIGN_ATOL, color="grey", linestyle=":", label="Absolute tolerance")
axes[2].set_ylabel("Standardized-input units; zeros shown at 1e-17")
add_panel_labels(axes)
fig.savefig(FIG / "02_backend_comparison.png", dpi=180)
fig.savefig(FIG / "02_backend_comparison.pdf")
plt.show()

case_name = "nonlinear_x_to_y"
age = cases[case_name]["age"]
window = age >= 24.0
reference = results[(case_name, "GPDC_cpu")]["residuals"][1]
fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True, constrained_layout=True)
for backend in backends:
    if (case_name, backend) not in results:
        continue
    residual = results[(case_name, backend)]["residuals"][1]
    axes[0].plot(age[window], residual[window], color=colors[backend], label=backend, linewidth=0.9, alpha=0.8)
    if backend.startswith("aligned"):
        axes[1].plot(age[window], (residual - reference)[window], color=colors[backend], label=backend)
axes[0].set_ylabel("Y residual\n(standardized-input units)")
axes[0].set_title("Nonlinear input: fixed X(t-1), Y(t) conditional test")
axes[0].legend(ncol=2, fontsize=9)
axes[1].set_ylabel("Aligned minus current GPDC")
axes[1].set_xlabel("Age (kyr BP)")
axes[1].invert_xaxis()
axes[1].legend(fontsize=9)
add_panel_labels(axes)
fig.savefig(FIG / "03_residual_alignment.png", dpi=180)
fig.savefig(FIG / "03_residual_alignment.pdf")
plt.show()


## 9. Read the result without changing the main study
Stock GPDCtorch differences combine model fitting, precision and backend
effects. The aligned CPU/CUDA comparisons isolate the fixed-model migration
more closely. Even successful agreement here does not validate all condition
sets, PCMCI+ search paths, chronology inputs or 32-worker/GPU scheduling.

The next decision should follow the regression errors and measured times below.
Do not switch the main pipeline merely because the six p-value decisions agree.


In [ ]:
aligned = comparison[comparison.backend.str.startswith("aligned")]
display(aligned.groupby("backend").agg(
    completed_tests=("case", "count"),
    largest_mean_error=("mean_max_abs_error", "max"),
    largest_residual_error=("residual_max_abs_error", "max"),
    all_within_tolerance=("regression_within_tolerance", "all"),
    largest_score_change=("score_difference", lambda values: np.max(np.abs(values))),
    largest_p_change=("p_difference", lambda values: np.max(np.abs(values))),
))
display(summary.groupby("backend").agg(completed_tests=("seconds", "count"),
                                        median_seconds=("seconds", "median"),
                                        total_seconds=("seconds", "sum")))
print("Saved tables and full regression arrays:", OUT)
print("Failures:", len(run_info["failed_tests"]))
if len(aligned) == len(cases) * (2 if HAS_CUDA else 1) and aligned.regression_within_tolerance.all():
    print("The aligned fixed model matches the reference within the stated regression tolerance.")
    print("Exact score agreement:", bool(aligned.score_difference.eq(0).all()))
    print("Exact p-value agreement:", bool(aligned.p_difference.eq(0).all()))
print("Parameter migration requires float64, frozen unit RBF/noise, zero mean and latent prediction.")
print("Default trained GPDCtorch is a different fit. Full PCMCI+ and production throughput remain untested.")
